# Sprint 13 Phase 3b — DoRA Persona Adapter Training (Kaggle T4)

**Goal**: Fine-tune Qwen2.5-7B-Instruct dengan DoRA rank-16 untuk 5 persona SIDIX (UTZ/ABOO/OOMAR/ALEY/AYMAN).

**Input**: `persona_qa_train.jsonl` + `persona_qa_val.jsonl` (output dari merge_persona_dataset.py).

**Output**: 1 LoRA adapter terlatih multi-persona (atau 5 adapter terpisah dengan loop). Upload ke HuggingFace `Tiranyx/sidix-dora-persona-v1`.

**Hardware**: Kaggle T4 16GB × 1. ~3-5 jam untuk 1500×5=7500 pairs × 3 epoch.

**Bos action items** (sebelum run):
1. Upload `persona_qa_train.jsonl` + `persona_qa_val.jsonl` ke Kaggle dataset (public atau private).
2. Set Kaggle Secrets: `HF_TOKEN` (write access ke org Tiranyx).
3. Pilih mode: SINGLE_ADAPTER (1 adapter multi-persona) atau PER_PERSONA_ADAPTER (5 adapter).

## 1 · Setup environment

In [ ]:
# No quantization — P100 incompat dgn modern bnb. bf16 + gradient checkpointing
# cukup untuk Qwen 7B + DoRA rank-16 di 16GB GPU.
!pip install -q --upgrade peft trl accelerate datasets huggingface_hub transformers
!pip show peft trl 2>&1 | grep -E 'Name|Version'


In [ ]:
import os, json, torch, glob
from pathlib import Path
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

BASE_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
OUTPUT_DIR = '/kaggle/working/sidix-dora-persona-v1'

_train_candidates = glob.glob('/kaggle/input/**/persona_qa_train.jsonl', recursive=True)
assert _train_candidates, 'dataset not found'
DATA_DIR = os.path.dirname(_train_candidates[0])
TRAIN_PATH = os.path.join(DATA_DIR, 'persona_qa_train.jsonl')
VAL_PATH = os.path.join(DATA_DIR, 'persona_qa_val.jsonl')

print(f'CUDA: {torch.cuda.is_available()}, device: {torch.cuda.get_device_name(0)}')
print(f'GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Train: {TRAIN_PATH} exists={os.path.exists(TRAIN_PATH)}')
print(f'Val:   {VAL_PATH} exists={os.path.exists(VAL_PATH)}')


## 2 · Load dataset (Alpaca → ChatML)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def to_chatml(row):
    """Alpaca → ChatML format. Persona di metadata BUKAN prompt.
    Model belajar voice persona TANPA cue eksplisit di system/user message."""
    messages = [
        {'role': 'user', 'content': row['instruction']},
        {'role': 'assistant', 'content': row['output']},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {'text': text}

train_ds = load_dataset('json', data_files=TRAIN_PATH, split='train').map(to_chatml, remove_columns=['instruction','input','output','metadata'])
val_ds = load_dataset('json', data_files=VAL_PATH, split='train').map(to_chatml, remove_columns=['instruction','input','output','metadata'])
print(f'train={len(train_ds)} val={len(val_ds)}')
print('Sample:', train_ds[0]['text'][:300])

## 3 · Load base model (4-bit) + DoRA config

In [ ]:
# bf16 load (no 4-bit quantization — P100 incompat dgn modern bitsandbytes)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
model.config.use_cache = False
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    use_dora=True,
    target_modules=['q_proj','k_proj','v_proj','o_proj'],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print(f'GPU mem after model load: {torch.cuda.memory_allocated() / 1e9:.2f} / {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')


## 4 · Training loop (SFTTrainer)

In [ ]:
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    optim='adamw_torch',
    bf16=True,
    max_seq_length=512,
    dataset_text_field='text',
    packing=False,
    save_strategy='epoch',
    eval_strategy='epoch',
    logging_steps=20,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
)
trainer.train()


## 5 · Save adapter + upload ke HuggingFace

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Saved to {OUTPUT_DIR}')

In [ ]:
# Skip auto-HF-upload dari Kaggle kernel (avoid Kaggle Secret requirement).
# Adapter tersimpan di /kaggle/working/ → auto-uploaded ke kernel output.
# Sesi post-training: download via `kaggle kernels output mighan/sidix-dora-train-v1`
# lalu upload ke HF dari VPS (HF_TOKEN sudah di env file VPS).

import os
print(f'Adapter saved to: {OUTPUT_DIR}')
print(f'Kernel output: https://www.kaggle.com/code/mighan/sidix-dora-train-v1/output')
print()
print('Next step (post-training, dari VPS):')
print('  kaggle kernels output mighan/sidix-dora-train-v1 -p ./adapter_dl')
print('  hf upload Tiranyx/sidix-dora-persona-v1 ./adapter_dl --repo-type model')
print()
print('=== Output files ===')
for root, dirs, files in os.walk(OUTPUT_DIR):
    for f in files:
        p = os.path.join(root, f)
        size_mb = os.path.getsize(p) / 1024 / 1024
        print(f'  {p}: {size_mb:.2f} MB')


## 6 · Quick smoke test (5 persona signature)

In [ ]:
# Generate 1 sample per persona untuk verify voice distinct
prompts = [
    'Bagaimana cara debug memory leak di Python?',
    'Cara bikin landing page yang convert?',
    'Framework prioritisasi feature backlog?',
    'Perbedaan transformer dan state-space model?',
    'Bagaimana hadapi feedback negatif dari atasan?',
]

model.eval()
for p in prompts:
    msg = [{'role': 'user', 'content': p}]
    text = tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=200, temperature=0.7, do_sample=True, top_p=0.9)
    response = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f'\nQ: {p}\nA: {response[:300]}\n' + '='*80)